## Reproducibility & AI-assistance disclosure

Part of `dislocation-speech-translation-fr-en`, companion to the M2 thesis *Dislocation under Translation: A Corpus Study of Whisper and XLS-R on Spontaneous Spoken French* (Zoé de Vries, Université Paris Cité, 2026; supervised by Prof. Ballier).

AI-assistance disclosure: explanatory comments and markdown were drafted with the help of a large language model (LLM) and reviewed by the author.

Pipeline order: `01_whisper` -> `02_xlsr` -> `03_align` -> `04_score`.

> Provenance note: the committed `data/dislocation_translations.csv` is the hand-verified alignment used in the thesis. This notebook documents the alignment method; re-running it over `segment_translations.csv` reproduces the spans but may differ from the committed file on some dislocation-level outputs (notably `whisper_en_cont`), which were finalised in a spreadsheet workflow. The committed CSV is canonical.

# Aligning dislocations to model outputs

Runs anywhere (pandas only).

For each annotated dislocation in `dislocation_test_set.csv`, find the corresponding Whisper segment(s) in `segment_translations.csv` and concatenate the FR/EN outputs of both models over that span. Output: `dislocation_translations.csv`, ready for scoring (notebook 04).

Method: temporal window (-5 s / +60 s) around the CFPP timestamp, anchor = segment with the highest content-word overlap with the citation, extension to the right (up to 3 segments) while overlap persists. The asymmetric window reflects the fact that CFPP timestamps mark the start of the speech turn, which precedes the dislocation.

In [ ]:
# Paths.

DISLOC_PATH       = 'data/dislocation_test_set.csv'
TRANSLATIONS_PATH = 'data/segment_translations.csv'
OUTPUT_PATH       = 'data/dislocation_translations.csv'

In [ ]:
import pandas as pd
import re

In [ ]:
# Load the test set (79 dislocations) and the segment grid (874 rows).

disloc = pd.read_csv(DISLOC_PATH)
csv    = pd.read_csv(TRANSLATIONS_PATH)

print(f'Dislocations loaded: {len(disloc)} rows')
print(f'Segments loaded:     {len(csv)} rows')

In [ ]:
# Text helpers for the content-word fuzzy match.

STOPWORDS = set('''
le la les un une des de du d à a au aux et ou est c'est ça je tu il elle on nous vous
ils elles que qui quoi pas ne se s en y me te dans pour
'''.split())

def normalize(text):
    """Lowercase, strip punctuation, normalise whitespace."""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def content_overlap_ratio(citation, segment_text):
    """Fraction of the citation's content words present in the segment."""
    content_words = [w for w in normalize(citation).split()
                     if w not in STOPWORDS and len(w) > 1]
    if not content_words:
        return 0.0
    seg_words = set(normalize(segment_text).split())
    matched = sum(1 for w in content_words if w in seg_words)
    return matched / len(content_words)

In [ ]:
# Align one dislocation to its segment span.

def align_dislocation(row, csv_df, window_back=5, window_fwd=60, max_extension=3):
    timestamp = row['Timestamp (s)']
    citation  = row['Citation contexte (FR)']

    window = csv_df[(csv_df['t_start'] >= timestamp - window_back) &
                    (csv_df['t_start'] <= timestamp + window_fwd)].copy()
    if len(window) == 0:
        return None

    window['overlap'] = window['whisper_fr'].apply(
        lambda x: content_overlap_ratio(citation, x))

    if window['overlap'].max() == 0:
        return None

    anchor_idx = window['overlap'].idxmax()
    anchor_pos = window.index.get_loc(anchor_idx)
    extended = [anchor_idx]

    for offset in range(1, max_extension + 1):
        next_pos = anchor_pos + offset
        if next_pos < len(window) and window.iloc[next_pos]['overlap'] > 0:
            extended.append(window.index[next_pos])
        else:
            break

    return csv_df.loc[extended].sort_values('t_start')

In [ ]:
# Run the alignment; unmatched rows get n_segs=0 and empty outputs.

rows = []
for _, drow in disloc.iterrows():
    matched = align_dislocation(drow, csv)

    base = {
        'ID': drow['ID'],
        'Speaker': drow['Speaker'],
        'Type': drow['Type'],
        'Citation contexte (FR)': drow['Citation contexte (FR)'],
        'Référence trad. CALQUE (EN)': drow['Référence trad. CALQUE (EN)'],
        'Référence trad. IDIOMATIC (EN)': drow['Référence trad. IDIOMATIC (EN)'],
    }

    if matched is None or len(matched) == 0:
        rows.append({**base,
            'n_segs': 0, 'segments': '', 'audio_start_s': None,
            'whisper_fr': '', 'xlsr_fr': '',
            'whisper_en_per_seg': '', 'whisper_en_cont': '',
            'xlsr_en': ''})
        continue

    rows.append({**base,
        'n_segs': len(matched),
        'segments': ' + '.join(matched['segment_id'].tolist()),
        'audio_start_s': round(float(matched.iloc[0]['t_start']), 2),
        'whisper_fr': ' '.join(' '.join(str(x).split()) for x in matched['whisper_fr'].fillna('')),
        'xlsr_fr':    ' '.join(' '.join(str(x).split()) for x in matched['xlsr_fr'].fillna('')),
        'whisper_en_per_seg': ' '.join(' '.join(str(x).split()) for x in matched['whisper_en_per_seg'].fillna('')),
        'whisper_en_cont': ' '.join(' '.join(str(x).split()) for x in matched['whisper_en_cont'].fillna('')),
        'xlsr_en':    ' '.join(' '.join(str(x).split()) for x in matched['xlsr_en'].fillna('')),
    })

aligned = pd.DataFrame(rows)

print(f'Dislocations processed: {len(aligned)}')
print(f'  - with >= 1 segment: {(aligned["n_segs"] > 0).sum()}')
print(f'  - no match:          {(aligned["n_segs"] == 0).sum()}')
print(f'\nSegments per dislocation: min={aligned["n_segs"].min()}, '
      f'max={aligned["n_segs"].max()}, mean={aligned["n_segs"].mean():.1f}')

In [ ]:
# Write the output CSV.

aligned.to_csv(OUTPUT_PATH, index=False, sep=',', encoding='utf-8')
print(f'CSV written: {OUTPUT_PATH}')
print(f'Columns ({len(aligned.columns)}): {list(aligned.columns)}')

In [ ]:
# Preview.

pd.set_option('display.max_colwidth', 80)
aligned[['ID', 'Speaker', 'Type', 'audio_start_s', 'n_segs', 'segments',
         'Citation contexte (FR)', 'whisper_fr']].head(10)